In [1]:
# ============================================================
# WILDDEEPFAKE CANONICAL TRAIN / VAL / TEST MANIFEST
# ============================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

EXTRACT_ROOT = (
    PROJECT_ROOT
    / "WildDeepfake"
    / "deepfake_in_the_wild"
    / "extracted"
)

OUT_DIR = (
    PROJECT_ROOT
    / "outputs_wilddeepfake"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

VAL_FRACTION = 0.10
SEED = 42


FOLDERS = [
    ("real_train", 0, "train"),
    ("fake_train", 1, "train"),
    ("real_test", 0, "test"),
    ("fake_test", 1, "test"),
]


rows = []
inventory = []


for folder, label, split in FOLDERS:

    root = (
        EXTRACT_ROOT
        / folder
    )

    if not root.exists():

        raise FileNotFoundError(
            root
        )


    paths = sorted(
        p
        for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTS
    )


    print(
        folder,
        "images:",
        len(paths),
    )


    for p in tqdm(
        paths,
        desc=folder,
        dynamic_ncols=True,
    ):

        relative = (
            p.relative_to(root)
        )

        parts = relative.parts

        archive_id = (
            parts[0]
            if len(parts) >= 1
            else p.parent.name
        )


        if len(parts) >= 3:

            inner_unit = "/".join(
                parts[1:-1]
            )

        elif len(parts) == 2:

            inner_unit = (
                archive_id
            )

        else:

            inner_unit = str(
                p.parent
                .relative_to(root)
            )


        sample_unit_id = (
            f"wilddeepfake::"
            f"{folder}::"
            f"{archive_id}::"
            f"{inner_unit}"
        )


        rows.append(
            {
                "frame_path": str(p),
                "image_path": str(p),

                "label": int(label),

                "label_name": (
                    "real"
                    if label == 0
                    else "fake"
                ),

                "split": split,

                "sample_unit_id":
                    sample_unit_id,

                "group_id":
                    sample_unit_id,

                "video_id":
                    sample_unit_id,

                "clip_id":
                    sample_unit_id,

                "method":
                    "wilddeepfake",

                "domain":
                    "wilddeepfake",

                "dataset":
                    "WildDeepfake",

                "dataset_partition":
                    f"wilddeepfake_{folder}",

                "top_level_folder":
                    folder,
            }
        )


df = pd.DataFrame(
    rows
)


# ============================================================
# Create validation split ONLY from official training data
# ============================================================

rng = np.random.default_rng(
    SEED
)

train_df = df[
    df["split"] == "train"
].copy()

test_df = df[
    df["split"] == "test"
].copy()


val_units = []


for label, group in train_df.groupby(
    "label"
):

    units = np.array(
        sorted(
            group[
                "sample_unit_id"
            ].unique()
        )
    )

    rng.shuffle(
        units
    )

    n_val = max(
        1,
        int(
            round(
                VAL_FRACTION
                * len(units)
            )
        ),
    )

    selected = (
        units[:n_val]
        .tolist()
    )

    val_units.extend(
        selected
    )


val_units = set(
    val_units
)


train_df.loc[
    train_df[
        "sample_unit_id"
    ].isin(val_units),
    "split",
] = "val"


df_final = pd.concat(
    [
        train_df,
        test_df,
    ],
    ignore_index=True,
)


# ============================================================
# SAFETY AUDIT
# ============================================================

print()
print("=" * 90)
print("FINAL SPLIT COUNTS")
print("=" * 90)

print(
    pd.crosstab(
        df_final["split"],
        df_final["label"],
    )
)


print()
print("Units:")

print(
    df_final.groupby(
        [
            "split",
            "label",
        ]
    )[
        "sample_unit_id"
    ]
    .nunique()
    .unstack(
        fill_value=0
    )
)


duplicate_paths = int(
    df_final[
        "image_path"
    ].duplicated().sum()
)


path_leakage = int(
    (
        df_final.groupby(
            "image_path"
        )["split"].nunique()
        > 1
    ).sum()
)


group_leakage = int(
    (
        df_final.groupby(
            "sample_unit_id"
        )["split"].nunique()
        > 1
    ).sum()
)


print()
print(
    "Duplicate paths:",
    duplicate_paths,
)

print(
    "Path leakage:",
    path_leakage,
)

print(
    "Group leakage:",
    group_leakage,
)


if duplicate_paths:
    raise RuntimeError(
        "Duplicate image paths detected."
    )

if path_leakage:
    raise RuntimeError(
        "Image-path split leakage detected."
    )

if group_leakage:
    raise RuntimeError(
        "Sample-unit split leakage detected."
    )


for split in [
    "train",
    "val",
    "test",
]:

    labels = set(
        df_final.loc[
            df_final["split"]
            == split,
            "label",
        ]
    )

    if labels != {0, 1}:

        raise RuntimeError(
            f"{split} does not "
            "contain both labels."
        )


# ============================================================
# SAVE
# ============================================================

canonical_path = (
    OUT_DIR
    / "wilddeepfake_manifest_canonical.csv"
)

train_path = (
    OUT_DIR
    / "wilddeepfake_train_manifest.csv"
)

val_path = (
    OUT_DIR
    / "wilddeepfake_val_manifest.csv"
)

test_path = (
    OUT_DIR
    / "wilddeepfake_test_manifest_canonical.csv"
)

audit_path = (
    OUT_DIR
    / "wilddeepfake_manifest_audit.csv"
)


df_final.to_csv(
    canonical_path,
    index=False,
)

df_final[
    df_final["split"] == "train"
].to_csv(
    train_path,
    index=False,
)

df_final[
    df_final["split"] == "val"
].to_csv(
    val_path,
    index=False,
)

df_final[
    df_final["split"] == "test"
].to_csv(
    test_path,
    index=False,
)


audit = {
    "dataset":
        "WildDeepfake",

    "source":
        "xingjunm/WildDeepfake",

    "validation_fraction":
        VAL_FRACTION,

    "seed":
        SEED,

    "total_frames":
        int(len(df_final)),

    "train_frames":
        int(
            (
                df_final["split"]
                == "train"
            ).sum()
        ),

    "val_frames":
        int(
            (
                df_final["split"]
                == "val"
            ).sum()
        ),

    "test_frames":
        int(
            (
                df_final["split"]
                == "test"
            ).sum()
        ),

    "duplicate_paths":
        duplicate_paths,

    "path_leakage":
        path_leakage,

    "group_leakage":
        group_leakage,
}


pd.DataFrame(
    [audit]
).to_csv(
    audit_path,
    index=False,
)


with open(
    canonical_path,
    "rb",
) as f:

    manifest_sha256 = (
        hashlib.sha256(
            f.read()
        ).hexdigest()
    )


(
    OUT_DIR
    / "wilddeepfake_manifest_sha256.txt"
).write_text(
    manifest_sha256
)


with open(
    OUT_DIR
    / "wilddeepfake_dataset_config.json",
    "w",
) as f:

    json.dump(
        {
            **audit,
            "manifest":
                str(
                    canonical_path
                ),
            "manifest_sha256":
                manifest_sha256,
        },
        f,
        indent=2,
    )


print()
print("=" * 90)
print("MANIFEST READY")
print("=" * 90)

print(
    canonical_path
)

print(
    "SHA256:",
    manifest_sha256,
)

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


real_train images: 381877


real_train: 100%|██████████| 381877/381877 [00:02<00:00, 151628.70it/s]


fake_train images: 632561


fake_train: 100%|██████████| 632561/632561 [00:04<00:00, 145472.09it/s]


real_test images: 58659


real_test: 100%|██████████| 58659/58659 [00:00<00:00, 172204.34it/s]


fake_test images: 107003


fake_test: 100%|██████████| 107003/107003 [00:00<00:00, 167764.57it/s]



FINAL SPLIT COUNTS
label       0       1
split                
test    58659  107003
train  346013  563531
val     35864   69030

Units:
label     0     1
split            
test    396   410
train  3069  2789
val     341   310

Duplicate paths: 0
Path leakage: 0
Group leakage: 0

MANIFEST READY
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
SHA256: 2f1743d8f6c984d6b882199a869dcf7aeb8c2dda53394b925b3c166454eddb2b


In [2]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

PLAN_PATH = (
    PROJECT_ROOT
    / "agrfn_wilddeepfake_ablation_plan.csv"
)

experiments = [
    {
        "experiment_name": "E0_late_fusion",
        "fusion_mode": "late",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": False,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.00,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.00,
        "lambda_unc": 0.00,
        "description": "GRFN-CSF late-fusion baseline",
    },

    {
        "experiment_name": "E1_average_fusion",
        "fusion_mode": "average",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": False,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.00,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.00,
        "lambda_unc": 0.00,
        "description": "Non-adaptive average fusion",
    },

    {
        "experiment_name": "E2_adaptive_scalar",
        "fusion_mode": "adaptive_scalar",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": False,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.01,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.00,
        "lambda_unc": 0.00,
        "description": "Fingerprint-conditioned scalar routing",
    },

    {
        "experiment_name": "E3_adaptive_film",
        "fusion_mode": "adaptive_film",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": False,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.01,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.00,
        "lambda_unc": 0.00,
        "description": "Fingerprint routing with FiLM modulation",
    },

    {
        "experiment_name": "E4_adaptive_transformer",
        "fusion_mode": "adaptive_transformer",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": False,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.01,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.00,
        "lambda_unc": 0.00,
        "description": "Fingerprint-conditioned transformer routing",
    },

    {
        "experiment_name": "E5_transformer_prototypes",
        "fusion_mode": "adaptive_transformer",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": True,
        "use_uncertainty": False,
        "lambda_aux": 0.05,
        "lambda_route": 0.01,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.10,
        "lambda_unc": 0.00,
        "description": "Transformer routing with class prototypes",
    },

    {
        "experiment_name": "E6_full_agrfn",
        "fusion_mode": "adaptive_transformer",
        "streams": "rgb,fingerprint,jpeg,freq",
        "use_prototypes": True,
        "use_uncertainty": True,
        "lambda_aux": 0.05,
        "lambda_route": 0.01,
        "lambda_cons": 0.02,
        "lambda_pred_cons": 0.02,
        "lambda_proto": 0.10,
        "lambda_unc": 0.10,
        "description": (
            "Full A-GRFN with transformer routing, "
            "prototypes, and learned uncertainty"
        ),
    },
]

plan = pd.DataFrame(experiments)

plan.to_csv(
    PLAN_PATH,
    index=False,
)

print("Saved:", PLAN_PATH)
print()
print(plan.to_string(index=False))

Saved: /home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_wilddeepfake_ablation_plan.csv

          experiment_name          fusion_mode                   streams  use_prototypes  use_uncertainty  lambda_aux  lambda_route  lambda_cons  lambda_pred_cons  lambda_proto  lambda_unc                                                               description
           E0_late_fusion                 late rgb,fingerprint,jpeg,freq           False            False        0.05          0.00         0.02              0.02           0.0         0.0                                             GRFN-CSF late-fusion baseline
        E1_average_fusion              average rgb,fingerprint,jpeg,freq           False            False        0.05          0.00         0.02              0.02           0.0         0.0                                               Non-adaptive average fusion
       E2_adaptive_scalar      adaptive_scalar rgb,fingerprint,jpeg,freq           False            False        0.0

In [7]:
from pathlib import Path
import shutil
from datetime import datetime

SCRIPT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution/"
    "agrfn_adaptive_routing_df40.py"
)

if not SCRIPT.exists():
    raise FileNotFoundError(SCRIPT)

# ------------------------------------------------------------
# Backup before changing anything
# ------------------------------------------------------------

backup = SCRIPT.with_name(
    "agrfn_adaptive_routing_df40_before_wild_io_patch.py"
)

if not backup.exists():
    shutil.copy2(SCRIPT, backup)
    print("Backup created:")
    print(backup)
else:
    print("Backup already exists:")
    print(backup)

text = SCRIPT.read_text(encoding="utf-8")


# ============================================================
# PATCH 1
# Add a flag that lets us skip the million-file exists scan
# ============================================================

old_cli = '''    parser.add_argument("--data_root", type=str, default=None,
                        help="Optional root prepended to relative image paths.")
'''

new_cli = '''    parser.add_argument("--data_root", type=str, default=None,
                        help="Optional root prepended to relative image paths.")

    parser.add_argument(
        "--skip_path_check",
        action="store_true",
        help=(
            "Skip the per-image os.path.exists scan during metadata loading. "
            "Use this only when the input manifest has already been audited "
            "and was created from files known to exist."
        ),
    )
'''

if old_cli not in text:
    raise RuntimeError(
        "Could not find CLI insertion point. "
        "No changes were written."
    )

text = text.replace(
    old_cli,
    new_cli,
    1,
)


# ============================================================
# PATCH 2
# Replace the expensive 1.18M-file existence sweep
# ============================================================

old_path_check = '''    # Keep only paths that exist.
    before = len(df)
    df = df[df["image_path"].apply(lambda p: isinstance(p, str) and os.path.exists(p))].copy()
    dropped = before - len(df)
    if len(df) == 0:
        raise RuntimeError("No usable image paths after path checking. Check --data_root and --path_col.")
    if dropped > 0:
        print(f"Dropped {dropped} rows with missing image paths.")
'''

new_path_check = '''    # Optional filesystem validation.
    #
    # Large pre-audited manifests such as WildDeepfake can contain more than
    # one million rows. Calling os.path.exists() once per row causes excessive
    # filesystem metadata I/O before training even begins.
    if args.skip_path_check:
        invalid_path_mask = ~df["image_path"].apply(
            lambda p: isinstance(p, str) and len(p.strip()) > 0
        )

        invalid_count = int(invalid_path_mask.sum())

        if invalid_count > 0:
            raise RuntimeError(
                f"Found {invalid_count} empty/invalid image paths "
                "while --skip_path_check was enabled."
            )

        print(
            f"Skipping per-file path existence scan for {len(df):,} rows "
            "(using pre-audited manifest).",
            flush=True,
        )

    else:
        before = len(df)

        df = df[
            df["image_path"].apply(
                lambda p: (
                    isinstance(p, str)
                    and os.path.exists(p)
                )
            )
        ].copy()

        dropped = before - len(df)

        if len(df) == 0:
            raise RuntimeError(
                "No usable image paths after path checking. "
                "Check --data_root and --path_col."
            )

        if dropped > 0:
            print(
                f"Dropped {dropped} rows with missing image paths.",
                flush=True,
            )
'''

if old_path_check not in text:
    raise RuntimeError(
        "Could not find existing path-check block. "
        "No changes were written."
    )

text = text.replace(
    old_path_check,
    new_path_check,
    1,
)


# ============================================================
# PATCH 3
# Replace slow repeated-dataframe filtering sampler
# ============================================================

start_marker = "def sample_frames_per_unit("
end_marker = (
    "\n\n# ============================================================\n"
    "# Multi-view forensic preprocessing"
)

start = text.find(start_marker)
end = text.find(end_marker, start)

if start == -1 or end == -1:
    raise RuntimeError(
        "Could not locate sample_frames_per_unit(). "
        "No changes were written."
    )

new_sampler = '''def sample_frames_per_unit(
    df,
    frames_per_unit=32,
    limit_units=None,
    seed=42,
    desc="Sampling",
    balanced_units=False,
):
    """Sample frames per unit without repeatedly scanning the full dataframe.

    The selected units and frame indices follow the same logic as the original
    implementation. The difference is that pandas groups the dataframe once
    instead of evaluating df[df["sample_unit_id"] == uid] for every unit.
    """

    if len(df) == 0:
        raise RuntimeError(
            f"{desc}: empty dataframe before sampling."
        )

    if frames_per_unit is None or frames_per_unit <= 0:
        out = df.copy().reset_index(drop=True)

        print(
            f"{desc}: no frame sampling. "
            f"frames={len(out)}, "
            f"units={out['sample_unit_id'].nunique()}, "
            f"real={(out['label'] == 0).sum()}, "
            f"fake={(out['label'] == 1).sum()}",
            flush=True,
        )

        return out

    # Same unit-label logic as before.
    unit_label = (
        df.groupby(
            "sample_unit_id",
            sort=True,
        )["label"]
        .first()
    )

    unit_ids = sorted(
        unit_label.index.tolist()
    )

    # --------------------------------------------------------
    # Preserve original limited-unit sampling behavior
    # --------------------------------------------------------

    if (
        limit_units is not None
        and len(unit_ids) > limit_units
    ):
        rng = np.random.default_rng(seed)

        if (
            balanced_units
            and len(unit_label.unique()) == 2
        ):
            chosen = []

            per_class = max(
                1,
                int(limit_units) // 2,
            )

            for label in [0, 1]:
                ids = sorted(
                    unit_label[
                        unit_label == label
                    ].index.tolist()
                )

                take = min(
                    per_class,
                    len(ids),
                )

                if take > 0:
                    chosen.extend(
                        rng.choice(
                            ids,
                            size=take,
                            replace=False,
                        ).tolist()
                    )

            remaining_slots = (
                int(limit_units)
                - len(chosen)
            )

            if remaining_slots > 0:
                remaining = sorted(
                    set(unit_ids)
                    - set(chosen)
                )

                if remaining:
                    take = min(
                        remaining_slots,
                        len(remaining),
                    )

                    chosen.extend(
                        rng.choice(
                            remaining,
                            size=take,
                            replace=False,
                        ).tolist()
                    )

            unit_ids = sorted(chosen)

        else:
            unit_ids = sorted(
                rng.choice(
                    unit_ids,
                    size=limit_units,
                    replace=False,
                ).tolist()
            )

    selected_units = set(unit_ids)

    # For debug runs, remove unselected units before grouping.
    # Full runs skip this copy because every unit is selected.
    if (
        len(selected_units)
        < df["sample_unit_id"].nunique()
    ):
        work_df = df[
            df["sample_unit_id"].isin(
                selected_units
            )
        ].copy()
    else:
        work_df = df

    # Group ONCE rather than scanning the dataframe once per unit.
    grouped = work_df.groupby(
        "sample_unit_id",
        sort=True,
    )

    sampled = []

    progress = tqdm(
        grouped,
        total=len(unit_ids),
        desc=desc,
        unit="unit",
        dynamic_ncols=True,

        # Visible but not noisy.
        mininterval=5.0,
        maxinterval=15.0,
        leave=True,
    )

    for uid, g in progress:

        # Defensive check for limited runs.
        if uid not in selected_units:
            continue

        # Preserve the original ordering before np.linspace sampling.
        g = (
            g.sort_values("image_path")
            .reset_index(drop=True)
        )

        if len(g["label"].unique()) != 1:
            raise ValueError(
                f"Mixed labels inside sample unit {uid}. "
                "Build a better group key."
            )

        if len(g) <= frames_per_unit:
            sampled.append(g)

        else:
            idx = np.linspace(
                0,
                len(g) - 1,
                frames_per_unit,
            ).astype(int)

            sampled.append(
                g.iloc[idx]
            )

    if not sampled:
        raise RuntimeError(
            "No frames found after sampling. "
            "Check split and paths."
        )

    out = pd.concat(
        sampled,
        axis=0,
    ).reset_index(drop=True)

    print(
        f"{desc}: "
        f"frames={len(out)}, "
        f"units={out['sample_unit_id'].nunique()}, "
        f"real={(out['label'] == 0).sum()}, "
        f"fake={(out['label'] == 1).sum()}",
        flush=True,
    )

    return out
'''

text = (
    text[:start]
    + new_sampler
    + text[end:]
)


# ============================================================
# PATCH 4
# Make training tqdm visible but less noisy
# ============================================================

old_train_bar = '''    pbar = tqdm(loader, desc=f"Train epoch {epoch}", dynamic_ncols=True)
'''

new_train_bar = '''    pbar = tqdm(
        loader,
        desc=f"Train epoch {epoch}",
        dynamic_ncols=True,
        mininterval=5.0,
        maxinterval=15.0,
        leave=True,
    )
'''

if old_train_bar not in text:
    raise RuntimeError(
        "Could not find training tqdm call."
    )

text = text.replace(
    old_train_bar,
    new_train_bar,
    1,
)


# ============================================================
# PATCH 5
# Make prediction tqdm visible but less noisy
# ============================================================

old_predict_bar = '''    pbar = tqdm(
        loader,
        desc=f"Predict {split_name}",
        dynamic_ncols=True,
    )
'''

new_predict_bar = '''    pbar = tqdm(
        loader,
        desc=f"Predict {split_name}",
        dynamic_ncols=True,
        mininterval=5.0,
        maxinterval=15.0,
        leave=True,
    )
'''

if old_predict_bar not in text:
    raise RuntimeError(
        "Could not find prediction tqdm call."
    )

text = text.replace(
    old_predict_bar,
    new_predict_bar,
    1,
)


# ============================================================
# PATCH 6
# Flush section headers through tee immediately
# ============================================================

old_print_stage = '''def print_stage(title):
    print("\\n" + "=" * 80)
    print(title)
    print("=" * 80)
'''

new_print_stage = '''def print_stage(title):
    print("\\n" + "=" * 80, flush=True)
    print(title, flush=True)
    print("=" * 80, flush=True)
'''

if old_print_stage not in text:
    raise RuntimeError(
        "Could not find print_stage()."
    )

text = text.replace(
    old_print_stage,
    new_print_stage,
    1,
)


# ------------------------------------------------------------
# Write patched script
# ------------------------------------------------------------

SCRIPT.write_text(
    text,
    encoding="utf-8",
)

print()
print("=" * 80)
print("PATCH COMPLETE")
print("=" * 80)
print("Updated:")
print(SCRIPT)
print()
print("Backup:")
print(backup)

Backup created:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_wild_io_patch.py

PATCH COMPLETE
Updated:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40.py

Backup:
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_wild_io_patch.py


In [8]:
import subprocess
import sys
import time
import importlib.util
from pathlib import Path
from types import SimpleNamespace

SCRIPT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution/"
    "agrfn_adaptive_routing_df40.py"
)

MANIFEST = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution/"
    "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv"
)

print("=" * 80)
print("1. SYNTAX CHECK")
print("=" * 80)

result = subprocess.run(
    [sys.executable, "-m", "py_compile", str(SCRIPT)],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print("FAILED")
    print(result.stderr)
    raise RuntimeError("Patched script has a syntax error.")

print("PASS - script compiles")


print("\n" + "=" * 80)
print("2. IMPORT PATCHED SCRIPT")
print("=" * 80)

spec = importlib.util.spec_from_file_location(
    "agrfn_patched",
    SCRIPT,
)

agrfn = importlib.util.module_from_spec(spec)
spec.loader.exec_module(agrfn)

print("PASS - script imports")


print("\n" + "=" * 80)
print("3. LOAD FULL WILDDEEPFAKE MANIFEST")
print("=" * 80)

args = SimpleNamespace(
    metadata_csv=str(MANIFEST),
    data_root=None,
    path_col=None,
    label_col=None,
    split_col=None,
    group_col=None,
    dataset_name="WildDeepfake",

    real_values="0,real",
    fake_values="1,fake",

    invert_labels=False,

    auto_split=False,
    val_ratio=0.10,
    test_ratio=0.10,
    seed=42,

    # Critical new option
    skip_path_check=True,
)

start = time.perf_counter()

df = agrfn.load_metadata(args)

metadata_seconds = time.perf_counter() - start

print()
print(f"Metadata load time: {metadata_seconds:.2f} seconds")
print(f"Rows: {len(df):,}")


print("\n" + "=" * 80)
print("4. FAST 32-FRAME SAMPLING TEST")
print("=" * 80)

expected_units = {
    "train": 5858,
    "val": 651,
    "test": 806,
}

sampled = {}

for split in ["train", "val", "test"]:

    split_df = (
        df[df["split"] == split]
        .reset_index(drop=True)
    )

    start = time.perf_counter()

    sampled[split] = agrfn.sample_frames_per_unit(
        split_df,
        frames_per_unit=32,
        limit_units=None,
        seed=42,
        desc=f"Sample {split}",
        balanced_units=False,
    )

    elapsed = time.perf_counter() - start

    actual_units = sampled[split]["sample_unit_id"].nunique()

    print(
        f"{split}: "
        f"{len(sampled[split]):,} frames, "
        f"{actual_units:,} units, "
        f"{elapsed:.2f} sec"
    )

    assert actual_units == expected_units[split], (
        f"{split} unit mismatch: "
        f"expected {expected_units[split]}, "
        f"got {actual_units}"
    )


print("\n" + "=" * 80)
print("5. SPLIT / LABEL CHECK")
print("=" * 80)

for split in ["train", "val", "test"]:

    sdf = sampled[split]

    units = (
        sdf[
            ["sample_unit_id", "label"]
        ]
        .drop_duplicates()
    )

    print(
        f"{split:5s} | "
        f"frames={len(sdf):,} | "
        f"units={len(units):,} | "
        f"real units={(units['label'] == 0).sum():,} | "
        f"fake units={(units['label'] == 1).sum():,}"
    )


print("\n" + "=" * 80)
print("VALIDATION PASSED")
print("=" * 80)
print(
    "The optimized metadata + sampling pipeline is ready "
    "for the full E0 run."
)

1. SYNTAX CHECK
PASS - script compiles

2. IMPORT PATCHED SCRIPT


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PASS - script imports

3. LOAD FULL WILDDEEPFAKE MANIFEST
Skipping per-file path existence scan for 1,180,100 rows (using pre-audited manifest).

Metadata loaded
Metadata CSV: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Path column: image_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 1180100
train: frames=909544, units=5858, real=346013, fake=563531
val: frames=104894, units=651, real=35864, fake=69030
test: frames=165662, units=806, real=58659, fake=107003

Metadata load time: 8.33 seconds
Rows: 1,180,100

4. FAST 32-FRAME SAMPLING TEST


Sample train: 100%|██████████| 5858/5858 [00:04<00:00, 1299.51unit/s]


Sample train: frames=187425, units=5858, real=98177, fake=89248
train: 187,425 frames, 5,858 units, 5.71 sec


Sample val: 100%|██████████| 651/651 [00:00<00:00, 1658.90unit/s]

Sample val: frames=20832, units=651, real=10912, fake=9920


val: 20,832 frames, 651 units, 0.46 sec


Sample test: 100%|██████████| 806/806 [00:00<00:00, 1330.79unit/s]

Sample test: frames=25792, units=806, real=12672, fake=13120
test: 25,792 frames, 806 units, 0.70 sec

5. SPLIT / LABEL CHECK
train | frames=187,425 | units=5,858 | real units=3,069 | fake units=2,789
val   | frames=20,832 | units=651 | real units=341 | fake units=310
test  | frames=25,792 | units=806 | real units=396 | fake units=410

VALIDATION PASSED
The optimized metadata + sampling pipeline is ready for the full E0 run.


In [3]:
# ?cd "/home/jovyan/Deepfake_Low Resource_Low Resolution"

!mkdir -p "outputs_agrfn_wilddeepfake_ablation/E0_late_fusion"

# Remove only the empty/stale console log from the failed attempts.
!rm -f "outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/train_console.log"

!python -u agrfn_adaptive_routing_df40.py \
  --metadata_csv "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv" \
  --out_dir "outputs_agrfn_wilddeepfake_ablation/E0_late_fusion" \
  --dataset_name WildDeepfake \
  --fusion_mode late \
  --streams rgb,fingerprint,jpeg,freq \
  --skip_path_check \
  --epochs 5 \
  --frames_per_video 32 \
  --image_size 224 \
  --batch_size 16 \
  --num_workers 2 \
  --feat_dim 128 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --lambda_stream 0.05 \
  --lambda_pred_cons 0.02 \
  --lambda_feat_cons 0.02 \
  --lambda_supcon 0.00 \
  --balance_strategy auto \
  --threshold_metric balanced_accuracy \
  --uncertainty_threshold 0.50 \
  --score_direction auto_val \
  --seed 42 \
  --amp \
  2>&1 | tee "outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/train_console.log"


GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_wilddeepfake_ablation/E0_late_fusion
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Streams: rgb,fingerprint,jpeg,freq
Skipping per-file path existence scan for 1,180,100 rows (using pre-audited manifest).

Metadata loaded
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Path column: image_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 1180100
train: frames=909544, units=5858, real=346013, fake=563531
val: frames=104894, units=651, real=35864, fake=69030
test: frames=165662, units=806, real=58659, fake=107003
Saved label audit samples to: outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/label_audit_samples.csv

Sampling frames per unit
Sample train: 100%|██████████| 5858/5858 [00:04<00:00, 1297.80unit/s]
Sample train: frames=187425, units=5858, real=98177, fake=89248

In [5]:
from pathlib import Path
import shutil

SCRIPT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution/"
    "agrfn_adaptive_routing_df40.py"
)

backup = SCRIPT.with_name(
    "agrfn_adaptive_routing_df40_before_resume_patch.py"
)

if not backup.exists():
    shutil.copy2(SCRIPT, backup)
    print("Backup created:", backup)

text = SCRIPT.read_text(encoding="utf-8")


# ============================================================
# PATCH 1: preserve training history when resuming
# ============================================================

old_resume = '''    start_epoch = 0
    best_val_auc = -1.0
    if args.resume is not None:
        print("Loading checkpoint:", args.resume)
        start_epoch, best_metric = load_checkpoint(model, optimizer, args.resume, device)
        if best_metric is not None:
            best_val_auc = float(best_metric)
        print("Resumed from epoch:", start_epoch, "best metric:", best_val_auc)

    history = []
'''

new_resume = '''    start_epoch = 0
    best_val_auc = -1.0
    history = []

    if args.resume is not None:
        print("Loading checkpoint:", args.resume)

        start_epoch, best_metric = load_checkpoint(
            model,
            optimizer,
            args.resume,
            device,
        )

        if best_metric is not None:
            best_val_auc = float(best_metric)

        # Preserve previous epoch history when continuing a run.
        history_path = (
            Path(args.out_dir)
            / "training_history.csv"
        )

        if history_path.exists():
            previous_history = pd.read_csv(
                history_path
            )

            # Keep only epochs up through the checkpoint
            # being resumed.
            if "epoch" in previous_history.columns:
                previous_history = previous_history[
                    previous_history["epoch"]
                    <= start_epoch
                ].copy()

            history = (
                previous_history
                .to_dict("records")
            )

            # Recover the actual best validation AUC from
            # history. This also protects against an older
            # checkpoint carrying a stale best_metric value.
            auc_col = "val_frame_AUCROC"

            if (
                auc_col in previous_history.columns
                and len(previous_history) > 0
            ):
                history_best = float(
                    previous_history[
                        auc_col
                    ].max()
                )

                best_val_auc = max(
                    best_val_auc,
                    history_best,
                )

            print(
                f"Loaded {len(history)} previous "
                "training-history rows."
            )

        print(
            "Resumed from epoch:",
            start_epoch,
            "best metric:",
            best_val_auc,
        )
'''

if old_resume not in text:
    raise RuntimeError(
        "Could not locate resume/history block."
    )

text = text.replace(
    old_resume,
    new_resume,
    1,
)


# ============================================================
# PATCH 2: make latest checkpoint store current best metric
# ============================================================

old_checkpoint = '''            save_checkpoint(
                model,
                optimizer,
                epoch,
                args,
                Path(args.out_dir) / "latest_checkpoint.pt",
                best_metric=best_val_auc,
            )

            if val_frame_auc > best_val_auc:
                best_val_auc = float(val_frame_auc)
                save_checkpoint(
                    model,
                    optimizer,
                    epoch,
                    args,
                    Path(args.out_dir) / "best_checkpoint.pt",
                    best_metric=best_val_auc,
                )
                print(f"New best val frame AUCROC: {best_val_auc:.4f}")
'''

new_checkpoint = '''            if val_frame_auc > best_val_auc:
                best_val_auc = float(val_frame_auc)

                save_checkpoint(
                    model,
                    optimizer,
                    epoch,
                    args,
                    Path(args.out_dir)
                    / "best_checkpoint.pt",
                    best_metric=best_val_auc,
                )

                print(
                    f"New best val frame AUCROC: "
                    f"{best_val_auc:.4f}"
                )

            # Save latest AFTER updating the best metric so
            # resume bookkeeping is always current.
            save_checkpoint(
                model,
                optimizer,
                epoch,
                args,
                Path(args.out_dir)
                / "latest_checkpoint.pt",
                best_metric=best_val_auc,
            )
'''

if old_checkpoint not in text:
    raise RuntimeError(
        "Could not locate checkpoint block."
    )

text = text.replace(
    old_checkpoint,
    new_checkpoint,
    1,
)

SCRIPT.write_text(
    text,
    encoding="utf-8",
)

print("=" * 80)
print("RESUME PATCH COMPLETE")
print("=" * 80)
print(SCRIPT)

Backup created: /home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40_before_resume_patch.py
RESUME PATCH COMPLETE
/home/jovyan/Deepfake_Low Resource_Low Resolution/agrfn_adaptive_routing_df40.py


In [7]:
# cd "/home/jovyan/Deepfake_Low Resource_Low Resolution"

%mkdir -p "outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/snapshot_epoch5"

%cp outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/*.csv \
   outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/snapshot_epoch5/ 2>/dev/null || true

%cp outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/best_checkpoint.pt \
   outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/snapshot_epoch5/best_checkpoint.pt

%cp outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/latest_checkpoint.pt \
   outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/snapshot_epoch5/latest_checkpoint.pt

In [9]:
!python -u agrfn_adaptive_routing_df40.py \
  --metadata_csv "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv" \
  --out_dir "outputs_agrfn_wilddeepfake_ablation/E0_late_fusion" \
  --dataset_name WildDeepfake \
  --fusion_mode late \
  --streams rgb,fingerprint,jpeg,freq \
  --skip_path_check \
  --epochs 10 \
  --frames_per_video 32 \
  --image_size 224 \
  --batch_size 16 \
  --num_workers 2 \
  --feat_dim 128 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --lambda_stream 0.05 \
  --lambda_pred_cons 0.02 \
  --lambda_feat_cons 0.02 \
  --lambda_supcon 0.00 \
  --balance_strategy auto \
  --threshold_metric balanced_accuracy \
  --uncertainty_threshold 0.50 \
  --score_direction auto_val \
  --seed 42 \
  --amp \
  --resume "outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/latest_checkpoint.pt" \
  2>&1 | tee -a "outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/train_console.log"


GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_wilddeepfake_ablation/E0_late_fusion
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Streams: rgb,fingerprint,jpeg,freq
Skipping per-file path existence scan for 1,180,100 rows (using pre-audited manifest).

Metadata loaded
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Path column: image_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 1180100
train: frames=909544, units=5858, real=346013, fake=563531
val: frames=104894, units=651, real=35864, fake=69030
test: frames=165662, units=806, real=58659, fake=107003
Saved label audit samples to: outputs_agrfn_wilddeepfake_ablation/E0_late_fusion/label_audit_samples.csv

Sampling frames per unit
Sample train: 100%|██████████| 5858/5858 [00:04<00:00, 1301.62unit/s]
Sample train: frames=187425, units=5858, real=98177, fake=89248

In [10]:
import pandas as pd
from pathlib import Path

folder = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution/"
    "outputs_agrfn_wilddeepfake_ablation/E0_late_fusion"
)

df = pd.read_csv(folder / "training_history.csv")

cols = [
    "epoch",
    "val_frame_Accuracy",
    "val_frame_Balanced Accuracy",
    "val_frame_AUCROC",
    "val_video_Accuracy",
    "val_video_Balanced Accuracy",
    "val_video_AUCROC",
]

print(df[cols].to_string(index=False))

print("\nBest frame AUC epoch:")
best_idx = df["val_frame_AUCROC"].idxmax()

print(
    df.loc[
        best_idx,
        ["epoch", "val_frame_AUCROC", "val_video_AUCROC"]
    ]
)

 epoch  val_frame_Accuracy  val_frame_Balanced Accuracy  val_frame_AUCROC  val_video_Accuracy  val_video_Balanced Accuracy  val_video_AUCROC
     1            0.761377                     0.760731          0.836882            0.791091                     0.790176          0.877060
     2            0.784514                     0.782758          0.861801            0.807988                     0.806452          0.899300
     3            0.801987                     0.803033          0.884725            0.832565                     0.834018          0.916867
     4            0.816868                     0.814892          0.892858            0.835637                     0.832551          0.923129
     5            0.825077                     0.824446          0.908813            0.855607                     0.856012          0.939514
     6            0.818836                     0.818031          0.904048            0.835637                     0.833138          0.935465
     7       

In [12]:
!cd "/home/jovyan/Deepfake_Low Resource_Low Resolution"

!mkdir -p "outputs_agrfn_wilddeepfake_ablation/E1_average_fusion"

!python -u agrfn_adaptive_routing_df40.py \
  --metadata_csv "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv" \
  --out_dir "outputs_agrfn_wilddeepfake_ablation/E1_average_fusion" \
  --dataset_name WildDeepfake \
  --fusion_mode average \
  --streams rgb,fingerprint,jpeg,freq \
  --skip_path_check \
  --epochs 10 \
  --frames_per_video 32 \
  --image_size 224 \
  --batch_size 16 \
  --num_workers 2 \
  --feat_dim 128 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --lambda_stream 0.05 \
  --lambda_pred_cons 0.02 \
  --lambda_feat_cons 0.02 \
  --lambda_supcon 0.00 \
  --balance_strategy auto \
  --threshold_metric balanced_accuracy \
  --uncertainty_threshold 0.50 \
  --score_direction auto_val \
  --seed 42 \
  --amp \
  2>&1 | tee "outputs_agrfn_wilddeepfake_ablation/E1_average_fusion/train_console.log"


GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_wilddeepfake_ablation/E1_average_fusion
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Streams: rgb,fingerprint,jpeg,freq
Skipping per-file path existence scan for 1,180,100 rows (using pre-audited manifest).

Metadata loaded
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Path column: image_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 1180100
train: frames=909544, units=5858, real=346013, fake=563531
val: frames=104894, units=651, real=35864, fake=69030
test: frames=165662, units=806, real=58659, fake=107003
Saved label audit samples to: outputs_agrfn_wilddeepfake_ablation/E1_average_fusion/label_audit_samples.csv

Sampling frames per unit
Sample train: 100%|██████████| 5858/5858 [00:04<00:00, 1312.64unit/s]
Sample train: frames=187425, units=5858, real=98177, fake

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Predict val: 100%|██████████| 1302/1302 [08:38<00:00,  2.51it/s]
Validation-tuned decision threshold: 0.45
New best val frame AUCROC: 0.9263

Validation metrics:
Level  Decision Threshold  Accuracy  Balanced Accuracy    AUPRC   AUCROC  EER (Equal Error Rate)  Precision (0)  Recall (0)   F1 (0)  Precision (1)  Recall (1)   F1 (1)  Uncertain Rate (%)  Predicted Real Count  Predicted Fake Count
Frame             0.45022  0.847782           0.846435 0.926528 0.926342                0.153996       0.841043    0.874725 0.857554       0.855847    0.818145 0.836572           31.590822                 11349                  9483
Video             0.45022  0.875576           0.874927 0.955200 0.954101                0.127566       0.875723    0.888563 0.882096       0.875410    0.861290 0.868293           32.411674                   346                   305

Final test evaluation
Loading best checkpoint for final test: outputs_agrfn_wilddeepfake_ablation/E1_average_fusion/best_checkpoint.pt
Pre

In [13]:
!cd "/home/jovyan/Deepfake_Low Resource_Low Resolution"

!mkdir -p "outputs_agrfn_wilddeepfake_ablation/E2_adaptive_scalar"

!python -u agrfn_adaptive_routing_df40.py \
  --metadata_csv "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv" \
  --out_dir "outputs_agrfn_wilddeepfake_ablation/E2_adaptive_scalar" \
  --dataset_name WildDeepfake \
  --fusion_mode adaptive_scalar \
  --streams rgb,fingerprint,jpeg,freq \
  --skip_path_check \
  --epochs 10 \
  --frames_per_video 32 \
  --image_size 224 \
  --batch_size 16 \
  --num_workers 2 \
  --feat_dim 128 \
  --controller_dim 128 \
  --routing_temperature 1.0 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --lambda_aux 0.05 \
  --lambda_route 0.01 \
  --lambda_cons 0.02 \
  --lambda_pred_cons 0.02 \
  --lambda_proto 0.00 \
  --lambda_unc 0.00 \
  --lambda_supcon 0.00 \
  --balance_strategy auto \
  --threshold_metric balanced_accuracy \
  --uncertainty_threshold 0.50 \
  --score_direction auto_val \
  --seed 42 \
  --amp \
  2>&1 | tee "outputs_agrfn_wilddeepfake_ablation/E2_adaptive_scalar/train_console.log"


GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_wilddeepfake_ablation/E2_adaptive_scalar
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Streams: rgb,fingerprint,jpeg,freq
Skipping per-file path existence scan for 1,180,100 rows (using pre-audited manifest).

Metadata loaded
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Path column: image_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 1180100
train: frames=909544, units=5858, real=346013, fake=563531
val: frames=104894, units=651, real=35864, fake=69030
test: frames=165662, units=806, real=58659, fake=107003
Saved label audit samples to: outputs_agrfn_wilddeepfake_ablation/E2_adaptive_scalar/label_audit_samples.csv

Sampling frames per unit
Sample train: 100%|██████████| 5858/5858 [00:04<00:00, 1284.11unit/s]
Sample train: frames=187425, units=5858, real=98177, fa

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Predict val: 100%|██████████| 1302/1302 [09:15<00:00,  2.35it/s]
Validation-tuned decision threshold: 0.52
New best val frame AUCROC: 0.8674

Validation metrics:
Level  Decision Threshold  Accuracy  Balanced Accuracy    AUPRC   AUCROC  EER (Equal Error Rate)  Precision (0)  Recall (0)  F1 (0)  Precision (1)  Recall (1)   F1 (1)  Uncertain Rate (%)  Predicted Real Count  Predicted Fake Count
Frame            0.521085  0.787778           0.787312 0.869148 0.867361                0.213421       0.797616    0.797104 0.79736       0.776972    0.777520 0.777246           50.499232                 10905                  9927
Video            0.521085  0.831029           0.830352 0.909908 0.903112                0.169062       0.834783    0.844575 0.83965       0.826797    0.816129 0.821429           53.302611                   345                   306
Predict val: 100%|██████████| 1302/1302 [09:10<00:00,  2.36it/s]
Validation-tuned decision threshold: 0.49
New best val frame AUCROC: 0.8836



In [14]:
!cd "/home/jovyan/Deepfake_Low Resource_Low Resolution"

!mkdir -p "outputs_agrfn_wilddeepfake_ablation/E3_adaptive_film"

!python -u agrfn_adaptive_routing_df40.py \
  --metadata_csv "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv" \
  --out_dir "outputs_agrfn_wilddeepfake_ablation/E3_adaptive_film" \
  --dataset_name WildDeepfake \
  --fusion_mode adaptive_film \
  --streams rgb,fingerprint,jpeg,freq \
  --skip_path_check \
  --epochs 10 \
  --frames_per_video 32 \
  --image_size 224 \
  --batch_size 16 \
  --num_workers 2 \
  --feat_dim 128 \
  --controller_dim 128 \
  --routing_temperature 1.0 \
  --film_scale 0.10 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --lambda_aux 0.05 \
  --lambda_route 0.01 \
  --lambda_cons 0.02 \
  --lambda_pred_cons 0.02 \
  --lambda_proto 0.00 \
  --lambda_unc 0.00 \
  --lambda_supcon 0.00 \
  --balance_strategy auto \
  --threshold_metric balanced_accuracy \
  --uncertainty_threshold 0.50 \
  --score_direction auto_val \
  --seed 42 \
  --amp \
  2>&1 | tee "outputs_agrfn_wilddeepfake_ablation/E3_adaptive_film/train_console.log"


GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_wilddeepfake_ablation/E3_adaptive_film
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Streams: rgb,fingerprint,jpeg,freq
Skipping per-file path existence scan for 1,180,100 rows (using pre-audited manifest).

Metadata loaded
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Path column: image_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 1180100
train: frames=909544, units=5858, real=346013, fake=563531
val: frames=104894, units=651, real=35864, fake=69030
test: frames=165662, units=806, real=58659, fake=107003
Saved label audit samples to: outputs_agrfn_wilddeepfake_ablation/E3_adaptive_film/label_audit_samples.csv

Sampling frames per unit
Sample train: 100%|██████████| 5858/5858 [00:04<00:00, 1250.28unit/s]
Sample train: frames=187425, units=5858, real=98177, fake=8

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Predict val: 100%|██████████| 1302/1302 [09:03<00:00,  2.40it/s]
Validation-tuned decision threshold: 0.28
New best val frame AUCROC: 0.8779

Validation metrics:
Level  Decision Threshold  Accuracy  Balanced Accuracy    AUPRC   AUCROC  EER (Equal Error Rate)  Precision (0)  Recall (0)   F1 (0)  Precision (1)  Recall (1)   F1 (1)  Uncertain Rate (%)  Predicted Real Count  Predicted Fake Count
Frame            0.277603  0.799779           0.798699 0.877941 0.877898                0.204202       0.801341    0.821389 0.811241       0.797968    0.776008 0.786835           33.414939                 11185                  9647
Video            0.277603  0.843318           0.842962 0.904900 0.908079                0.161437       0.850440    0.850440 0.850440       0.835484    0.835484 0.835484           32.104455                   341                   310
Predict val: 100%|██████████| 1302/1302 [09:07<00:00,  2.38it/s]8it/s, cls=0.3555, cons=0.1034, stream=0.5690, total=0.3869]
Validation-tun

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Predict val: 100%|██████████| 1302/1302 [07:30<00:00,  2.89it/s]
Validation-tuned decision threshold: 0.47
New best val frame AUCROC: 0.9172

Validation metrics:
Level  Decision Threshold  Accuracy  Balanced Accuracy    AUPRC   AUCROC  EER (Equal Error Rate)  Precision (0)  Recall (0)   F1 (0)  Precision (1)  Recall (1)   F1 (1)  Uncertain Rate (%)  Predicted Real Count  Predicted Fake Count
Frame            0.472736  0.841014           0.840762 0.914713 0.917223                0.160617       0.849779    0.846041 0.847906       0.831461    0.835484 0.833467           27.102535                 10864                  9968
Video            0.472736  0.881720           0.881232 0.947490 0.948936                0.115103       0.883721    0.891496 0.887591       0.879479    0.870968 0.875203           24.270353                   344                   307
Predict val: 100%|██████████| 1302/1302 [08:52<00:00,  2.45it/s]
Validation-tuned decision threshold: 0.42
New best val frame AUCROC: 0.918

In [15]:
!cd "/home/jovyan/Deepfake_Low Resource_Low Resolution"

!mkdir -p "outputs_agrfn_wilddeepfake_ablation/E4_adaptive_transformer"

!python -u agrfn_adaptive_routing_df40.py \
  --metadata_csv "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv" \
  --out_dir "outputs_agrfn_wilddeepfake_ablation/E4_adaptive_transformer" \
  --dataset_name WildDeepfake \
  --fusion_mode adaptive_transformer \
  --streams rgb,fingerprint,jpeg,freq \
  --skip_path_check \
  --epochs 10 \
  --frames_per_video 32 \
  --image_size 224 \
  --batch_size 16 \
  --num_workers 2 \
  --feat_dim 128 \
  --controller_dim 128 \
  --router_heads 4 \
  --router_layers 1 \
  --router_dropout 0.10 \
  --routing_temperature 1.0 \
  --film_scale 0.10 \
  --lr 1e-4 \
  --weight_decay 1e-4 \
  --lambda_aux 0.05 \
  --lambda_route 0.01 \
  --lambda_cons 0.02 \
  --lambda_pred_cons 0.02 \
  --lambda_proto 0.00 \
  --lambda_unc 0.00 \
  --lambda_supcon 0.00 \
  --balance_strategy auto \
  --threshold_metric balanced_accuracy \
  --uncertainty_threshold 0.50 \
  --score_direction auto_val \
  --seed 42 \
  --amp \
  2>&1 | tee "outputs_agrfn_wilddeepfake_ablation/E4_adaptive_transformer/train_console.log"


GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_wilddeepfake_ablation/E4_adaptive_transformer
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Streams: rgb,fingerprint,jpeg,freq
Skipping per-file path existence scan for 1,180,100 rows (using pre-audited manifest).

Metadata loaded
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Path column: image_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 1180100
train: frames=909544, units=5858, real=346013, fake=563531
val: frames=104894, units=651, real=35864, fake=69030
test: frames=165662, units=806, real=58659, fake=107003
Saved label audit samples to: outputs_agrfn_wilddeepfake_ablation/E4_adaptive_transformer/label_audit_samples.csv

Sampling frames per unit
Sample train: 100%|██████████| 5858/5858 [00:04<00:00, 1318.89unit/s]
Sample train: frames=187425, units=5858, real

In [ ]:
from pathlib import Path
import subprocess
import shlex

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

OUT_DIR = PROJECT_ROOT / "outputs_agrfn_wilddeepfake_ablation/E5"
LOG_PATH = OUT_DIR / "train_console_nohup.log"
PID_PATH = OUT_DIR / "train_nohup.pid"

OUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "python", "-u", "agrfn_adaptive_routing_df40.py",
    "--metadata_csv", "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv",
    "--out_dir", "outputs_agrfn_wilddeepfake_ablation/E5",
    "--dataset_name", "WildDeepfake",

    # Change this to whatever E5 is supposed to be.
    # Examples: late, adaptive_scalar, adaptive_film, adaptive_transformer
    "--fusion_mode", "adaptive_film",

    "--streams", "rgb,fingerprint,jpeg,freq",
    "--skip_path_check",
    "--epochs", "5",
    "--frames_per_video", "32",
    "--image_size", "224",
    "--batch_size", "16",
    "--num_workers", "0",
    "--feat_dim", "128",
    "--lr", "1e-4",
    "--weight_decay", "1e-4",
    "--lambda_stream", "0.05",
    "--lambda_pred_cons", "0.02",
    "--lambda_feat_cons", "0.02",
    "--lambda_supcon", "0.00",
    "--balance_strategy", "auto",
    "--threshold_metric", "balanced_accuracy",
    "--uncertainty_threshold", "0.50",
    "--score_direction", "auto_val",
    "--seed", "42",
    "--amp",
]

bash_cmd = (
    "cd " + shlex.quote(str(PROJECT_ROOT)) + " && "
    "PYTHONUNBUFFERED=1 nohup " + " ".join(shlex.quote(x) for x in cmd) +
    " > " + shlex.quote(str(LOG_PATH)) + " 2>&1 & echo $!"
)

proc = subprocess.run(
    ["bash", "-lc", bash_cmd],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

pid = proc.stdout.strip().splitlines()[-1]
PID_PATH.write_text(pid)

print("Launched A-GRFN E5 PID:", pid)
print("Log:", LOG_PATH)
print("PID file:", PID_PATH)

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_agrfn_wilddeepfake_ablation/E5"
LOG_PATH = OUT_DIR / "train_console_nohup.log"
PID_PATH = OUT_DIR / "train_nohup.pid"

pid = PID_PATH.read_text().strip() if PID_PATH.exists() else None
print("PID:", pid)

if pid:
    print(subprocess.run(
        ["bash", "-lc", f"ps -p {pid} -o pid,ppid,etime,stat,%cpu,%mem,rss,cmd || true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ).stdout)

print("\nGPU:")
print(subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout or "No GPU process found.")

print("\nOutput files:")
for name in [
    "latest_checkpoint.pt",
    "best_checkpoint.pt",
    "training_history.csv",
    "test_metrics.csv",
    "train_console_nohup.log",
]:
    p = OUT_DIR / name
    if p.exists():
        print(f"FOUND   {name:30s} size_mb={p.stat().st_size/1024**2:.3f} age_min={(time.time()-p.stat().st_mtime)/60:.1f}")
    else:
        print(f"MISSING {name}")

print("\nLog tail:")
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    for line in lines[-120:]:
        print(line)
else:
    print("No log yet.")

In [4]:
# ============================================================
# Launch A-GRFN E6 on WildDeepfake with nohup
# Faster settings: batch_size 32, num_workers 4
# WARNING: Do not run this together with E5 unless you accept instability.
# ============================================================

from pathlib import Path
import subprocess
import shlex

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")

OUT_DIR = PROJECT_ROOT / "outputs_agrfn_wilddeepfake_ablation/E6"
LOG_PATH = OUT_DIR / "train_console_nohup.log"
PID_PATH = OUT_DIR / "train_nohup.pid"

OUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    "python", "-u", "agrfn_adaptive_routing_df40.py",
    "--metadata_csv", "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv",
    "--out_dir", "outputs_agrfn_wilddeepfake_ablation/E6",
    "--dataset_name", "WildDeepfake",

    # Update this if E6 is meant to be a different mode.
    # Common choices: late, adaptive_scalar, adaptive_film, adaptive_transformer
    "--fusion_mode", "adaptive_transformer",

    "--streams", "rgb,fingerprint,jpeg,freq",
    "--skip_path_check",
    "--epochs", "5",
    "--frames_per_video", "32",
    "--image_size", "224",

    # Faster settings
    "--batch_size", "24",
    "--num_workers", "2",

    "--feat_dim", "128",
    "--lr", "1e-4",
    "--weight_decay", "1e-4",
    "--lambda_stream", "0.05",
    "--lambda_pred_cons", "0.02",
    "--lambda_feat_cons", "0.02",
    "--lambda_supcon", "0.00",
    "--balance_strategy", "auto",
    "--threshold_metric", "balanced_accuracy",
    "--uncertainty_threshold", "0.50",
    "--score_direction", "auto_val",
    "--seed", "42",
    "--amp",
]

print("=" * 100)
print("Checking existing heavy training processes")
print("=" * 100)

existing = subprocess.run(
    ["bash", "-lc", "ps aux | egrep 'agrfn_adaptive_routing_df40.py|train_dfdhr_manifest_wilddeepfake.py|gend_reimpl_ffpp.py|freqdebias_ffpp.py' | grep -v egrep || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip()

print(existing if existing else "No heavy training process found.")

print("\n" + "=" * 100)
print("Launching E6")
print("=" * 100)
print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))
print("Output:", OUT_DIR)
print("Log:", LOG_PATH)

# Prevent duplicate E6, but allow other experiments if you insist.
existing_e6 = subprocess.run(
    ["bash", "-lc", "ps aux | grep -i 'agrfn_adaptive_routing_df40.py' | grep 'outputs_agrfn_wilddeepfake_ablation/E6' | grep -v grep || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip()

if existing_e6:
    print("\nE6 is already running. Not launching duplicate:")
    print(existing_e6)
else:
    bash_cmd = (
        "cd " + shlex.quote(str(PROJECT_ROOT)) + " && "
        "PYTHONUNBUFFERED=1 nohup " + " ".join(shlex.quote(x) for x in cmd) +
        " > " + shlex.quote(str(LOG_PATH)) + " 2>&1 & echo $!"
    )

    proc = subprocess.run(
        ["bash", "-lc", bash_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    pid = proc.stdout.strip().splitlines()[-1]
    PID_PATH.write_text(pid)

    print("\nLaunched A-GRFN E6 PID:", pid)
    print("PID saved to:", PID_PATH)

Checking existing heavy training processes
jovyan       212  0.0  0.0   6672  2468 ?        S    00:06   0:00 grep -E agrfn_adaptive_routing_df40.py|train_dfdhr_manifest_wilddeepfake.py|gend_reimpl_ffpp.py|freqdebias_ffpp.py

Launching E6
Command:
python -u agrfn_adaptive_routing_df40.py --metadata_csv outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv --out_dir outputs_agrfn_wilddeepfake_ablation/E6 --dataset_name WildDeepfake --fusion_mode adaptive_transformer --streams rgb,fingerprint,jpeg,freq --skip_path_check --epochs 5 --frames_per_video 32 --image_size 224 --batch_size 24 --num_workers 2 --feat_dim 128 --lr 1e-4 --weight_decay 1e-4 --lambda_stream 0.05 --lambda_pred_cons 0.02 --lambda_feat_cons 0.02 --lambda_supcon 0.00 --balance_strategy auto --threshold_metric balanced_accuracy --uncertainty_threshold 0.50 --score_direction auto_val --seed 42 --amp
Output: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_wilddeepfake_ablation/E6
Log: /home/jovyan/Deepfake

In [3]:
from pathlib import Path
import subprocess
import pandas as pd
import time

PROJECT_ROOT = Path("/home/jovyan/Deepfake_Low Resource_Low Resolution")
OUT_DIR = PROJECT_ROOT / "outputs_agrfn_wilddeepfake_ablation/E6"
LOG_PATH = OUT_DIR / "train_console_nohup.log"
PID_PATH = OUT_DIR / "train_nohup.pid"

print("=" * 100)
print("A-GRFN E6 status")
print("=" * 100)

if PID_PATH.exists():
    pid = PID_PATH.read_text().strip()
    print("PID:", pid)

    print(subprocess.run(
        ["bash", "-lc", f"ps -p {pid} -o pid,ppid,etime,stat,%cpu,%mem,rss,cmd || true"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ).stdout)
else:
    print("No PID file found.")

print("\nAll A-GRFN jobs:")
print(subprocess.run(
    ["bash", "-lc", "ps aux | grep -i 'agrfn_adaptive_routing_df40.py' | grep -v grep || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout)

print("\nGPU:")
print(subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv,noheader,nounits || true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout or "No GPU process found.")

print("\nOutput files:")
for name in [
    "latest_checkpoint.pt",
    "best_checkpoint.pt",
    "training_history.csv",
    "test_metrics.csv",
    "train_console_nohup.log",
]:
    p = OUT_DIR / name
    if p.exists():
        print(f"FOUND   {name:30s} size_mb={p.stat().st_size/1024**2:.3f} age_min={(time.time()-p.stat().st_mtime)/60:.1f}")
    else:
        print(f"MISSING {name}")

print("\nHistory:")
for name in ["training_history.csv", "training_period_results.csv"]:
    p = OUT_DIR / name
    if p.exists():
        display(pd.read_csv(p).tail(10))
        break

print("\nLog tail:")
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(errors="replace").splitlines()
    print("Log lines:", len(lines))
    for line in lines[-120:]:
        print(line)
else:
    print("No log yet.")

A-GRFN E6 status
No PID file found.

All A-GRFN jobs:


GPU:
No GPU process found.

Output files:
MISSING latest_checkpoint.pt
MISSING best_checkpoint.pt
MISSING training_history.csv
MISSING test_metrics.csv
FOUND   train_console_nohup.log        size_mb=0.003 age_min=42.3

History:

Log tail:
Log lines: 69

GRFN-CSF GRFN-only training and evaluation
Device: cuda
Output directory: /home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_wilddeepfake_ablation/E6
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Streams: rgb,fingerprint,jpeg,freq
Skipping per-file path existence scan for 1,180,100 rows (using pre-audited manifest).

Metadata loaded
Metadata CSV: outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv
Path column: image_path
Label column: label
Split column: split
Group column: group_id
Usable frames: 1180100
train: frames=909544, units=5858, real=346013, fake=563531
val: frames=104894, units=651, real=35864, fake=69030
test: frames=165662, un

In [4]:
!pip install numpy transformers torchvision scikit-learn pandas

  Using cached numpy-2.4.6-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached transformers-5.16.1-py3-none-any.whl.metadata (32 kB)
  Using cached torchvision-0.29.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.7 kB)
  Using cached scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached pandas-3.0.5-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached regex-2026.9.3-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached typer-0.27.2-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached torch-2.14.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (37 kB)
  Using cached pillow-12.3.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached filelock-3.32.5-

In [1]:
print('test')

test


In [1]:
# ============================================================
# Resume A-GRFN E3 Adaptive FiLM on WildDeepfake
# Continue existing training through epoch 15
#
# IMPORTANT:
#   - Resumes from latest_checkpoint.pt
#   - Does NOT restart from epoch 1
#   - Keeps batch_size=16 for experimental consistency
#   - Uses num_workers=4 + AMP for better throughput
# ============================================================

from pathlib import Path
import subprocess
import shlex
import pandas as pd


# ============================================================
# Paths
# ============================================================

PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

OUT_DIR = (
    PROJECT_ROOT
    / "outputs_agrfn_wilddeepfake_ablation"
    / "E3_adaptive_film"
)

CHECKPOINT_PATH = (
    OUT_DIR
    / "latest_checkpoint.pt"
)

HISTORY_PATH = (
    OUT_DIR
    / "training_history.csv"
)

LOG_PATH = (
    OUT_DIR
    / "resume_epoch11_15_nohup.log"
)

PID_PATH = (
    OUT_DIR
    / "resume_epoch11_15_nohup.pid"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Verify existing E3 run
# ============================================================

print("=" * 100)
print("VERIFYING EXISTING E3 ADAPTIVE FILM RUN")
print("=" * 100)

print("\nOutput directory:")
print(OUT_DIR)

print("\nCheckpoint:")
print(CHECKPOINT_PATH)


if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Cannot resume E3 because latest_checkpoint.pt "
        f"was not found:\n{CHECKPOINT_PATH}"
    )


print("\nCheckpoint found: YES")


# ============================================================
# Check latest recorded epoch
# ============================================================

last_epoch = None

if HISTORY_PATH.exists():

    history = pd.read_csv(
        HISTORY_PATH
    )

    if (
        len(history) > 0
        and "epoch" in history.columns
    ):

        last_epoch = int(
            pd.to_numeric(
                history["epoch"],
                errors="coerce",
            )
            .dropna()
            .max()
        )

        print("\nLatest epoch in training_history.csv:")
        print(last_epoch)

else:

    print(
        "\nWARNING: training_history.csv was not found."
    )


if last_epoch is not None:

    if last_epoch >= 15:

        raise RuntimeError(
            f"E3 already contains epoch {last_epoch}. "
            "There is nothing to resume through epoch 15."
        )

    print(
        f"\nExpected next epoch: {last_epoch + 1}"
    )


# ============================================================
# Build resume command
# ============================================================

cmd = [
    "python",
    "-u",
    "agrfn_adaptive_routing_df40.py",

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------

    "--metadata_csv",
    "outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv",

    "--out_dir",
    "outputs_agrfn_wilddeepfake_ablation/E3_adaptive_film",

    "--dataset_name",
    "WildDeepfake",

    # --------------------------------------------------------
    # E3 architecture
    # --------------------------------------------------------

    "--fusion_mode",
    "adaptive_film",

    "--streams",
    "rgb,fingerprint,jpeg,freq",

    "--feat_dim",
    "128",

    "--controller_dim",
    "128",

    "--film_scale",
    "0.10",

    # --------------------------------------------------------
    # Resume existing run
    # --------------------------------------------------------

    "--resume",
    str(CHECKPOINT_PATH),

    # Total epoch target.
    #
    # Because checkpoint is already at epoch 10,
    # this should run epochs 11, 12, 13, 14, 15.
    "--epochs",
    "15",

    # --------------------------------------------------------
    # Dataset sampling
    # --------------------------------------------------------

    "--skip_path_check",

    "--frames_per_video",
    "32",

    "--image_size",
    "224",

    # --------------------------------------------------------
    # Performance settings
    # --------------------------------------------------------
    #
    # Keep batch size identical to epochs 1-10.
    #
    # Increasing num_workers can improve GPU utilization by
    # preparing RGB/JPEG/frequency inputs concurrently.
    # --------------------------------------------------------

    "--batch_size",
    "16",

    "--num_workers",
    "2",

    "--amp",

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    "--lr",
    "1e-4",

    "--weight_decay",
    "1e-4",

    # --------------------------------------------------------
    # Loss configuration
    #
    # Keep these consistent with the original ablation setup.
    # --------------------------------------------------------

    "--lambda_stream",
    "0.05",

    "--lambda_pred_cons",
    "0.02",

    "--lambda_feat_cons",
    "0.02",

    "--lambda_supcon",
    "0.00",

    # --------------------------------------------------------
    # Evaluation / balancing
    # --------------------------------------------------------

    "--balance_strategy",
    "auto",

    "--threshold_metric",
    "balanced_accuracy",

    "--uncertainty_threshold",
    "0.50",

    "--score_direction",
    "auto_val",

    "--seed",
    "42",
]


# ============================================================
# Check for existing heavy processes
# ============================================================

print("\n" + "=" * 100)
print("CHECKING EXISTING HEAVY TRAINING PROCESSES")
print("=" * 100)

existing = subprocess.run(
    [
        "bash",
        "-lc",
        (
            "ps aux | "
            "egrep "
            "'agrfn_adaptive_routing_df40.py|"
            "train_dfdhr_manifest_wilddeepfake.py|"
            "gend_reimpl_ffpp.py|"
            "freqdebias_ffpp.py' "
            "| grep -v egrep || true"
        ),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip()

print(
    existing
    if existing
    else "No heavy training process found."
)


# ============================================================
# Show command before launching
# ============================================================

print("\n" + "=" * 100)
print("RESUMING E3 ADAPTIVE FILM")
print("=" * 100)

print("\nCommand:\n")

print(
    " ".join(
        shlex.quote(x)
        for x in cmd
    )
)

print("\nOutput:")
print(OUT_DIR)

print("\nResume checkpoint:")
print(CHECKPOINT_PATH)

print("\nLog:")
print(LOG_PATH)


# ============================================================
# Prevent duplicate E3 run
# ============================================================

existing_e3 = subprocess.run(
    [
        "bash",
        "-lc",
        (
            "ps aux | "
            "grep -i 'agrfn_adaptive_routing_df40.py' | "
            "grep 'outputs_agrfn_wilddeepfake_ablation/E3_adaptive_film' | "
            "grep -v grep || true"
        ),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
).stdout.strip()


if existing_e3:

    print("\n" + "!" * 100)
    print("E3 IS ALREADY RUNNING")
    print("!" * 100)

    print(
        "\nNot launching another copy."
    )

    print("\nExisting process:")
    print(existing_e3)


else:

    # ========================================================
    # nohup launch
    # ========================================================

    bash_cmd = (
        "cd "
        + shlex.quote(
            str(PROJECT_ROOT)
        )
        + " && "
        + "PYTHONUNBUFFERED=1 "
        + "nohup "
        + " ".join(
            shlex.quote(x)
            for x in cmd
        )
        + " > "
        + shlex.quote(
            str(LOG_PATH)
        )
        + " 2>&1 "
        + "& echo $!"
    )


    proc = subprocess.run(
        [
            "bash",
            "-lc",
            bash_cmd,
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )


    output = (
        proc.stdout
        .strip()
        .splitlines()
    )


    if not output:

        raise RuntimeError(
            "nohup launch returned no PID."
        )


    pid = output[-1].strip()


    PID_PATH.write_text(
        pid
    )


    print("\n" + "=" * 100)
    print("E3 RESUME LAUNCHED")
    print("=" * 100)

    print("\nPID:")
    print(pid)

    print("\nPID saved to:")
    print(PID_PATH)

    print("\nLog:")
    print(LOG_PATH)

    print(
        "\nYou can close the notebook/browser. "
        "The process will continue under nohup."
    )

VERIFYING EXISTING E3 ADAPTIVE FILM RUN

Output directory:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_wilddeepfake_ablation/E3_adaptive_film

Checkpoint:
/home/jovyan/Deepfake_Low Resource_Low Resolution/outputs_agrfn_wilddeepfake_ablation/E3_adaptive_film/latest_checkpoint.pt

Checkpoint found: YES

Latest epoch in training_history.csv:
10

Expected next epoch: 11

CHECKING EXISTING HEAVY TRAINING PROCESSES
jovyan       744  0.0  0.0   6672  2332 ?        S    00:33   0:00 grep -E agrfn_adaptive_routing_df40.py|train_dfdhr_manifest_wilddeepfake.py|gend_reimpl_ffpp.py|freqdebias_ffpp.py

RESUMING E3 ADAPTIVE FILM

Command:

python -u agrfn_adaptive_routing_df40.py --metadata_csv outputs_wilddeepfake/wilddeepfake_manifest_canonical.csv --out_dir outputs_agrfn_wilddeepfake_ablation/E3_adaptive_film --dataset_name WildDeepfake --fusion_mode adaptive_film --streams rgb,fingerprint,jpeg,freq --feat_dim 128 --controller_dim 128 --film_scale 0.10 --resume '/home/jovyan/De

In [ ]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

LOG_PATH = (
    PROJECT_ROOT
    / "outputs_agrfn_wilddeepfake_ablation"
    / "E3_adaptive_film"
    / "resume_epoch11_15_nohup.log"
)

result = subprocess.run(
    [
        "tail",
        "-80",
        str(LOG_PATH),
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)

In [6]:
from pathlib import Path
import subprocess
import time
import pandas as pd

PROJECT_ROOT = Path(
    "/home/jovyan/Deepfake_Low Resource_Low Resolution"
)

OUT_DIR = (
    PROJECT_ROOT
    / "outputs_agrfn_wilddeepfake_ablation"
    / "E3_adaptive_film"
)

PID_PATH = OUT_DIR / "resume_epoch11_15_nohup.pid"
LOG_PATH = OUT_DIR / "resume_epoch11_15_nohup.log"
HISTORY_PATH = OUT_DIR / "training_history.csv"
CKPT_PATH = OUT_DIR / "latest_checkpoint.pt"

print("=" * 80)
print("E3 STATUS CHECK")
print("=" * 80)

# ------------------------------------------------------------
# Process
# ------------------------------------------------------------

pid = None

if PID_PATH.exists():
    pid = PID_PATH.read_text().strip()
    print("\nPID:", pid)

    result = subprocess.run(
        [
            "bash",
            "-lc",
            f"ps -p {pid} -o pid,ppid,stat,etime,%cpu,%mem,cmd || true"
        ],
        capture_output=True,
        text=True,
    )

    print("\nPROCESS STATUS:")
    print(
        result.stdout.strip()
        if result.stdout.strip()
        else "PROCESS NOT FOUND"
    )
else:
    print("\nPID file not found.")


# ------------------------------------------------------------
# Log age
# ------------------------------------------------------------

if LOG_PATH.exists():
    age = time.time() - LOG_PATH.stat().st_mtime

    print("\nLOG LAST UPDATED:")
    print(f"{age:.1f} seconds ago")

    print("\nLAST LOG LINES:")

    result = subprocess.run(
        ["tail", "-5", str(LOG_PATH)],
        capture_output=True,
        text=True,
    )

    print(result.stdout)


# ------------------------------------------------------------
# GPU process
# ------------------------------------------------------------

print("\nGPU:")
result = subprocess.run(
    [
        "nvidia-smi",
        "--query-compute-apps=pid,used_memory",
        "--format=csv,noheader"
    ],
    capture_output=True,
    text=True,
)

print(
    result.stdout.strip()
    if result.stdout.strip()
    else "No GPU compute process detected."
)


# ------------------------------------------------------------
# Latest completed epoch
# ------------------------------------------------------------

if HISTORY_PATH.exists():

    history = pd.read_csv(HISTORY_PATH)

    if "epoch" in history.columns:
        print("\nLATEST COMPLETED EPOCH:")
        print(
            int(
                pd.to_numeric(
                    history["epoch"],
                    errors="coerce"
                ).dropna().max()
            )
        )


# ------------------------------------------------------------
# Checkpoint modification time
# ------------------------------------------------------------

if CKPT_PATH.exists():
    ckpt_age = time.time() - CKPT_PATH.stat().st_mtime

    print("\nLATEST CHECKPOINT UPDATED:")
    print(f"{ckpt_age / 60:.1f} minutes ago")

E3 STATUS CHECK

PID file not found.

LOG LAST UPDATED:
3076.5 seconds ago

LAST LOG LINES:

Training

Train epoch 11:  48%|████▊     | 5675/11714 [43:09<47:35,  2.11it/s, cls=0.1940, cons=0.1539, stream=0.4983, total=0.2228]

GPU:
No GPU compute process detected.

LATEST COMPLETED EPOCH:
10

LATEST CHECKPOINT UPDATED:
13694.9 minutes ago
